# USGS National Water-Quality Assessment (NAWQA) Project: Pesticide National Synthesis Project


* Provides annual agricultural pesticide use estimates by county, based on farm surveys of pesticide use and estimates of harvested crop acres. [LINK](https://water.usgs.gov/nawqa/pnsp/usage/maps/county-level/)

* The website also provides maps of agricultural pesticide use on a finer scale which are created by allocating the county-level estimates to agricultural land within each county. A graph accompanies each map and shows annual national use by major crop for the mapped pesticide for each year. More on how these are generated: [LINK](https://water.usgs.gov/nawqa/pnsp/usage/maps/about.php)

* Available time range: 1992–2019

* Consists of a tab-separated file (txt) for each year from 1992 to 2012, and an additional metadata file (xml) for 2013-2017 range, 2018, and 2019. The metadata contains details on how the dataset was produced and the FIPS code used.

* The estimate will be NaN if there were not enough data to make an estimate -- which I assume will be more prevalent for older years.

* Though I'm ignoring this for the sake of time, since FIPS code had slight changes over time, albeit limited to small number of states, the best practice would be to grab FIPS code chart from the census for each publication year. (e.g. [2017 FIPS](https://www.census.gov/geographies/reference-files/2017/demo/popest/2017-fips.html))


In [ ]:
# load relevant packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# load estimates from 2013 to 2017
df = pd.read_csv('https://www.sciencebase.gov/catalog/file/get/5e95c12282ce172707f2524e?f=__disk__62%2F83%2Fd3%2F6283d3501f1028b1ccc3976ea2e6de848bc2fef8&allowOpen=true',sep='\t')
df.head()

In [ ]:
# 2018 and 2019 have separate txt files
df2018 = pd.read_csv('https://www.sciencebase.gov/catalog/file/get/6081a706d34e8564d686618e?f=__disk__58%2F6a%2Fed%2F586aed9a844eac0174a0600c8a7293ec4cda0265&allowOpen=true',sep='\t')
df2018.head()

In [ ]:
df2019 = pd.read_csv('https://www.sciencebase.gov/catalog/file/get/6081a924d34e8564d68661a1?f=__disk__08%2F42%2Fcd%2F0842cdac3a7d8b5056645a4dc08d1da96ad4e0b7&allowOpen=true',sep='\t')
df2019.head()

In [ ]:
# for years 1992-2012, you can grab the text files from the url directly from their website
# could potentially loop over across all years and append to the same dataframe, 
# granted that they use the same format of urls and just change the year
for year in range(1992,2013):
    url = 'https://water.usgs.gov/nawqa/pnsp/usage/maps/county-level/PesticideUseEstimates/EPest.county.estimates.'+str(year)+'.txt'
    this_df = pd.read_csv(url,sep='\t')
    df = pd.concat([df,this_df])

In [ ]:
df = pd.concat([df,df2018,df2019])

In [ ]:
years = df['YEAR'].unique()
years.sort()
print(years)

In [ ]:
df.describe()

In [ ]:
df.info(verbose=True)

In [ ]:
# There's counties with the same code in different states, so we'll need to create another column that is state+county code so that we get unique county codes.
df['STATE_FIPS_CODE'] = df['STATE_FIPS_CODE'].apply(lambda x: f"{x:02d}")
df['COUNTY_FIPS_CODE'] = df['COUNTY_FIPS_CODE'].apply(lambda x: f"{x:03d}")
df['STATE_COUNTY_CODE'] = df['STATE_FIPS_CODE']+df['COUNTY_FIPS_CODE']

In [ ]:
df.head()

In [ ]:
# try plotting some things
# adding a mean estimate column
df['EPEST_MEAN_KG'] = df[['EPEST_LOW_KG', 'EPEST_HIGH_KG']].mean(axis=1)
df_groups = df.groupby('YEAR')['EPEST_MEAN_KG'].mean()


In [ ]:
# average pesticide use across all counties and all compounds
df_groups.plot(kind='bar')
plt.xlabel('Year')
plt.ylabel('Average Pesticide Use (kg)')
plt.show()

In [ ]:
# pick a random county and plot cumulative pesticide use across all available years
counties = df['STATE_COUNTY_CODE'].unique()
this_county = counties[np.random.randint(low=0,high=len(counties)+1)]
df_county = df[df['STATE_COUNTY_CODE']==this_county].groupby('COMPOUND')['EPEST_MEAN_KG'].sum()
df_county.plot.pie()

In [ ]:
# plotting a single compound use across years for this county
this_compound = '2,4-D'
df_subset = df[(df['STATE_COUNTY_CODE']==this_county) & (df['COMPOUND']==this_compound)]
sorted_df = df_subset.sort_values(by='YEAR')
sorted_df

In [ ]:
plt.plot(sorted_df['YEAR'],sorted_df['EPEST_MEAN_KG'])
plt.title('Average '+this_compound+' use in county '+str(this_county))
plt.xlabel('Year')
plt.ylabel('Estimated pesticide use (kg)')
plt.show()